# Fulltext and Hybrid Search via MCP

This notebook introduces **fulltext keyword search** and **agent-driven hybrid search** that combines vector similarity with keyword matching — all executed through MCP.

**Learning Objectives:**
- Use fulltext search operators (fuzzy, wildcard, boolean) through MCP
- Understand when fulltext search outperforms vector search and vice versa
- Combine both approaches with custom `@tool` wrappers for agent-driven hybrid search
- Inspect raw tool call inputs and outputs for debugging

**How it works:**
1. Fulltext search uses Apache Lucene indexes for exact keyword matching — no embedding needed
2. Vector search uses embeddings for semantic similarity — finds meaning, not exact words
3. Hybrid search runs both and lets the agent synthesize from combined results
4. Custom `@tool` wrappers encapsulate embedding generation so the agent never sees raw float arrays

**Prerequisites:**
- `../CONFIG.txt` configured with `MODEL_ID`, `REGION`, `MCP_GATEWAY_URL`, and `MCP_ACCESS_TOKEN`
- Data loaded into Neo4j with vector embeddings and a `search_chunks` fulltext index

In [ ]:
%pip install -U langgraph langchain-aws langchain-mcp-adapters mcp nest-asyncio -q

## 1. Configuration and Setup

Import utilities from `lib/` to handle configuration and MCP connection boilerplate.

In [ ]:
import json
import sys
import nest_asyncio

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

nest_asyncio.apply()

sys.path.insert(0, '..')

from lib.data_utils import get_embedding, BedrockConfig
from lib.mcp_utils import MCPConnection
import os
from dotenv import load_dotenv

# Load config for LLM setup
load_dotenv('../CONFIG.txt')
MODEL_ID = os.getenv('MODEL_ID')
REGION = os.getenv('REGION', 'us-east-1')

if MODEL_ID and MODEL_ID.startswith('us.anthropic.'):
    BASE_MODEL_ID = MODEL_ID.replace('us.anthropic.', 'anthropic.')
elif MODEL_ID and MODEL_ID.startswith('global.anthropic.'):
    BASE_MODEL_ID = MODEL_ID.replace('global.anthropic.', 'anthropic.')
else:
    BASE_MODEL_ID = None

# Initialize LLM
llm_kwargs = {'model': MODEL_ID, 'region_name': REGION, 'temperature': 0}
if BASE_MODEL_ID:
    llm_kwargs['base_model_id'] = BASE_MODEL_ID
llm = ChatBedrockConverse(**llm_kwargs)

# Connect to MCP server
mcp = await MCPConnection.create('../CONFIG.txt')

# Test embedding function
test_emb = get_embedding('test')
print(f'Model:     {MODEL_ID}')
print(f'Embedding: {len(test_emb)} dimensions')
print('Setup complete!')

## 2. Fulltext Search Agent

Fulltext search uses Apache Lucene indexes to match exact keywords in chunk text. Unlike vector search, no embedding is needed — the agent passes a search term directly to the fulltext index.

This is useful for queries with specific entity names, ticker symbols, or technical terms that vector search might not rank highly.

In [ ]:
FULLTEXT_PROMPT = """You are a retrieval assistant that performs fulltext keyword search against a Neo4j database containing SEC 10-K filing data.

You have MCP tools to execute Cypher queries. Use the fulltext index on Chunk text:

CALL db.index.fulltext.queryNodes('search_chunks', $search_term)
YIELD node, score
RETURN node.text AS text, score
ORDER BY score DESC
LIMIT $limit

SEARCH OPERATORS (use these in the search term string):
- Fuzzy: append ~ to handle typos (e.g., 'revnue~' matches 'revenue')
- Wildcard: append * for prefix matching (e.g., 'risk*' matches 'risks', 'risky')
- Boolean AND: both terms required (e.g., 'revenue AND growth')
- Boolean NOT: exclude terms (e.g., 'revenue NOT decline')

For each result, show the Lucene relevance score and a preview of the chunk text."""

fulltext_agent = create_react_agent(llm, mcp.get_tools(), prompt=FULLTEXT_PROMPT)
print('Fulltext agent ready!')

## 3. Basic Fulltext Search

Search for chunks containing a specific keyword. No embedding is needed — the agent passes the term directly to the Lucene index.

In [ ]:
async def fulltext_search(term: str, limit: int = 5):
    """Run a fulltext keyword search through the MCP agent."""
    print(f'Search term: "{term}"')
    print('-' * 60)

    message = f"Search for chunks containing '{term}'. Use limit={limit}."
    result = await fulltext_agent.ainvoke({'messages': [HumanMessage(content=message)]})
    print(result['messages'][-1].content)
    return result


result = await fulltext_search('revenue')

## 4. Search Operators

Fulltext indexes support several operators that vector search cannot replicate:

| Operator | Syntax | Purpose |
|----------|--------|---------|
| Fuzzy | `term~` | Handles typos and spelling variations |
| Wildcard | `term*` | Matches partial terms |
| Boolean AND | `term1 AND term2` | Both terms must appear |
| Boolean NOT | `term1 NOT term2` | First term present, second absent |

In [ ]:
# Fuzzy search — handles typos
result = await fulltext_search('revnue~', limit=3)

In [ ]:
# Wildcard search — prefix matching
result = await fulltext_search('risk*', limit=3)

In [ ]:
# Boolean AND — both terms must appear
result = await fulltext_search('revenue AND growth', limit=3)

## 5. Fulltext + Graph Traversal

Combine fulltext keyword matching with graph traversal to pull in document metadata and connected entities. This is the same enrichment pattern from notebook 02, but applied to fulltext matches instead of vector matches.

In [ ]:
FULLTEXT_GRAPH_PROMPT = """You are a retrieval assistant that performs fulltext search with graph context against a Neo4j database containing SEC 10-K filing data.

When given a search term, use this Cypher to find keyword matches WITH entity context:

CALL db.index.fulltext.queryNodes('search_chunks', $search_term)
YIELD node, score
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (company:Company)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies,
     collect(DISTINCT product.name) AS products
RETURN node.text AS text, score, doc.name AS document, companies, products
ORDER BY score DESC
LIMIT $limit

For each result, show the score, document name, matched text, and any connected companies or products."""

fulltext_graph_agent = create_react_agent(llm, mcp.get_tools(), prompt=FULLTEXT_GRAPH_PROMPT)
print('Fulltext + graph agent ready!')

In [ ]:
async def fulltext_graph_search(term: str, limit: int = 5):
    """Run fulltext search with graph traversal."""
    print(f'Search term: "{term}" (with graph context)')
    print('-' * 60)

    message = f"Search for chunks containing '{term}' with graph context. Use limit={limit}."
    result = await fulltext_graph_agent.ainvoke({'messages': [HumanMessage(content=message)]})
    print(result['messages'][-1].content)
    return result


result = await fulltext_graph_search('iPhone')

## 6. Vector vs Fulltext — When to Use Each

| Query Type | Fulltext | Vector |
|------------|----------|--------|
| Specific company name ("Apple Inc") | Strong exact match | May match other companies with similar descriptions |
| Ticker symbol ("AAPL") | Exact keyword match | Embedding may not capture ticker semantics |
| Conceptual question ("supply chain risks") | Only if exact words appear | Finds semantically related content regardless of wording |
| Fuzzy or partial terms ("revnue~") | Built-in fuzzy matching | No fuzzy capability |

Neither approach is strictly better. The next section combines both.

## 7. Agent-Driven Hybrid Search with `@tool` Wrappers

Instead of serializing a 1024-dimensional embedding array into the agent's prompt, we wrap embedding generation and MCP query execution inside custom `@tool` functions. The agent calls `vector_search("Apple products")` — a clean interface that hides the embedding internals.

This is the recommended LangChain pattern: custom `@tool` functions that encapsulate retrieval logic, giving the agent high-level search capabilities without exposing implementation details.

In [ ]:
@tool
async def vector_search(query: str, top_k: int = 5) -> str:
    """Search for semantically similar chunks using vector embeddings.
    Use this for conceptual or semantic queries where exact words may differ."""
    # Step 1: Embed the query
    embedding = get_embedding(query)

    # Step 2: Vector search via MCP
    return await mcp.execute_query(f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, {json.dumps(embedding)})
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN node.text AS text, score, doc.name AS document
        ORDER BY score DESC
    """)


@tool
async def fulltext_search_tool(term: str, limit: int = 5) -> str:
    """Search for chunks containing specific keywords.
    Use this for exact terms, company names, tickers, or partial matches.
    Supports operators: fuzzy (term~), wildcard (term*), AND, NOT."""
    return await mcp.execute_query(f"""
        CALL db.index.fulltext.queryNodes('search_chunks', '{term}')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN node.text AS text, score, doc.name AS document
        ORDER BY score DESC
        LIMIT {limit}
    """)


print(f'Custom tools created:')
print(f'  - vector_search: {vector_search.description}')
print(f'  - fulltext_search_tool: {fulltext_search_tool.description}')

In [ ]:
HYBRID_PROMPT = """You are a financial analysis assistant that combines vector (semantic) and fulltext (keyword) search to answer questions about SEC 10-K filings.

You have two search tools:
1. vector_search: Finds semantically similar content (good for conceptual questions)
2. fulltext_search_tool: Finds exact keyword matches (good for specific names, terms, tickers)

HYBRID SEARCH STRATEGY:
For comprehensive results, run BOTH search tools with the same query, then synthesize an answer from the combined results. This gives you the benefits of both semantic understanding and keyword precision.

When the query contains specific names or terms (like "Apple", "AAPL"), fulltext search may find more precise matches. When the query is conceptual ("supply chain risks"), vector search captures semantic meaning.

For the best results, always run both searches and compare what each found."""

# Agent gets ONLY the custom tools — never sees raw embeddings or MCP tools
hybrid_agent = create_react_agent(llm, [vector_search, fulltext_search_tool], prompt=HYBRID_PROMPT)
print('Hybrid agent ready!')

In [ ]:
async def hybrid_search(query: str):
    """Run hybrid search using both vector and fulltext tools."""
    print(f'Question: "{query}"')
    print('=' * 60)

    message = f"Answer this question using BOTH vector search and fulltext search: {query}"
    result = await hybrid_agent.ainvoke({'messages': [HumanMessage(content=message)]})
    print(f'\n{result["messages"][-1].content}')
    return result


result = await hybrid_search("What are Apple's key risk factors?")

In [ ]:
result = await hybrid_search('Which companies face cybersecurity-related risks?')

## 8. Understanding Alpha Tuning

The `HybridRetriever` in the neo4j-graphrag library (Lab 6) uses an **alpha** parameter to mathematically balance vector and fulltext scores:

```
combined_score = alpha * vector_score + (1 - alpha) * fulltext_score
```

| Alpha | Behavior |
|-------|---------|
| 1.0 | Pure vector (semantic only) |
| 0.7 | Favor semantic, supplement with keywords |
| 0.5 | Equal weight to both |
| 0.3 | Favor keywords, supplement with semantic |
| 0.0 | Pure fulltext (keywords only) |

**Through MCP, the agent approximates this by running both searches and using its judgment to weigh the results.** This is less mathematically precise than the library approach, but it teaches the underlying concept: different queries benefit from different balances of semantic and keyword matching.

For programmatic alpha tuning with automatic score normalization, see Lab 6's `HybridRetriever` and `HybridCypherRetriever`.

## 9. Inspecting Tool Calls

In MCP-based retrieval, the agent formats results before presenting them. To see the raw data — what query was sent and what Neo4j returned — inspect the tool call messages in the agent's message history.

In [ ]:
def show_tool_calls(result):
    """Display raw tool call inputs and outputs from an agent run."""
    for msg in result['messages']:
        # Tool call requests (from the agent)
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for call in msg.tool_calls:
                print(f"\n{'=' * 60}")
                print(f"TOOL CALL: {call['name']}")
                args = call.get('args', {})
                for key, value in args.items():
                    val_str = str(value)
                    if len(val_str) > 200:
                        print(f"  {key}: {val_str[:200]}... [{len(val_str)} chars]")
                    else:
                        print(f"  {key}: {val_str}")

        # Tool responses (from Neo4j via MCP)
        if hasattr(msg, 'name') and msg.name:
            content = str(msg.content)
            print(f"\nRESPONSE ({msg.name}):")
            print(f"  {content[:500]}{'...' if len(content) > 500 else ''}")


# Run a search and inspect the raw tool calls
result = await hybrid_search('What products does Apple offer?')
print('\n\n=== RAW TOOL CALLS ===')
show_tool_calls(result)

## Summary

You built three search approaches across the Lab 4 notebooks, all executed through MCP:

| Notebook | Search Method | Graph Context | Embedding Needed |
|----------|--------------|---------------|------------------|
| 01 | Vector | No | Yes (in prompt) |
| 02 | Vector | Document + Chunks + Entities | Yes (in prompt) |
| 03 | Fulltext | Optional | No |
| 03 | Hybrid (`@tool`) | Optional | Yes (hidden in tool) |

The `@tool` wrapper pattern is the key improvement: embedding generation and MCP query execution are encapsulated inside custom tools, so the agent calls `vector_search("Apple products")` instead of receiving a giant float array. This is the recommended LangChain pattern for production retrieval agents.

For the programmatic `HybridRetriever` and `HybridCypherRetriever` classes with automatic alpha tuning, see [Lab 6](../Lab_6_GraphRAG/).

---

**Next:** Continue to [Lab 6](../Lab_6_GraphRAG/) for direct library-based retrieval, or [Lab 7](../Lab_7_Neo4j_MCP_Agent/) to build full MCP agents.